In [11]:
# Loading the environment variables 
import os
from dotenv import load_dotenv,find_dotenv
load_dotenv(find_dotenv())


True

In [12]:
# creating an instance of chat model with name model 
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    model="openai/gpt-oss-120b",
    api_key=os.getenv("Free_API_KEY"),
    temperature=0.0,
)

In [13]:
# Creating an instance of embedding model with name embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
)


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
doc_path="./data/be-good.txt"
loader=TextLoader(doc_path)
docs=loader.load()

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=text_splitter.split_documents(docs)

vector_db=Chroma.from_documents(documents=chunks,embedding=embeddings)
retriever=vector_db.as_retriever()


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [15]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("user","{input}")
    ]
)

In [16]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
answer_chain=create_stuff_documents_chain(llm,prompt)
rag_chain=create_retrieval_chain(retriever,answer_chain)

In [ ]:
output = rag_chain.invoke({"input": "What is this article about?"})
print(output)

c:\Users\pande\AppData\Local\pypoetry\Cache\virtualenvs\conversational-rag-app-rQOhCcmB-py3.11\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{'input': 'What is this article about?', 'context': [Document(metadata={'source': './data/be-good.txt'}, page_content="April 2008(This essay is derived from a talk at the 2008 Startup School.)About a month after we started Y Combinator we came up with the\nphrase that became our motto: Make something people want.  We've\nlearned a lot since then, but if I were choosing now that's still\nthe one I'd pick.Another thing we tell founders is not to worry too much about the\nbusiness model, at least at first.  Not because making money is\nunimportant, but because it's so much easier than building something\ngreat.A couple weeks ago I realized that if you put those two ideas\ntogether, you get something surprising.  Make something people want.\nDon't worry too much about making money.  What you've got is a\ndescription of a charity.When you get an unexpected result like this, it could either be a\nbug or a new discovery.  Either businesses aren't supposed to be\nlike charities, and we've prov

Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=c285c7e5-a341-4f57-851e-115eafb97328,id=c285c7e5-a341-4f57-851e-115eafb97328; trace=c285c7e5-a341-4f57-851e-115eafb97328,id=73a7e911-df3d-4efd-9f22-bddcc2e8e433; trace=c285c7e5-a341-4f57-851e-115eafb97328,id=524690db-e1fd-458f-86c6-9e97f1558cda; trace=c285c7e5-a341-4f57-851e-115eafb97328,id=876a79c9-d998-4824-bdb8-3780cfc873b7; trace=c285c7e5-a341-4f57-851e-115eafb97328,id=1ae80502-eb83-4361-adc0-4818c7655c58; trace=c285c7e5-a341-4f57-851e-115eafb97328,id=12f6255d-a41b-43e7-b499-16db1b4e609f; trace=c285c7e5-a341-4f57-851e-115eafb97328,id=726be9ab-9f60-4a5c-83d8-e73cd3d6053b; trace=c285c7e5-a341-4f57-851e-115eafb97328,id=34252f40-8d59-45ce-bcca-8535cc61c745; trace=c285c7e5-a341-4f57-851e-115eafb97328,id=f32f0d31-e

## Create a ChatPromptTemplate able to contextualize inputs
* Goal: put the input in context and re-phrase it so we have a contextualized input.
* We will define a new system prompt that instructs the LLM in how to contextualize the input.
* Our new ChatPromptTemplate will include:
    * The new system prompt.
    * MessagesPlaceholder, a placeholder used to pass the list of messages included in the chat_history.

In [19]:
from langchain_core.prompts import MessagesPlaceholder
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextual_prompt=ChatPromptTemplate.from_messages(
    [
        ("system",contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("user","{input}"),
    ]
)

## Create a Retriever aware of the memory
* We will build our new retriever with create_history_aware_retriever that uses the contextualized input to get a contextualized response.

In [20]:
from langchain.chains import create_history_aware_retriever
history_aware_retriever=create_history_aware_retriever(
    llm,retriever,contextual_prompt
)

In [21]:
qa_prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
         MessagesPlaceholder("chat_history"),
         ("human","{input}")

    ]
)

answer1_chain=create_stuff_documents_chain(llm,qa_prompt)
rag1_chain=create_retrieval_chain(history_aware_retriever,answer1_chain)

Creating a simple Chat history Manual way

In [ ]:
from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
question="What is this article about?"
ai_msg_1=rag1_chain.invoke({"input":question,"chat_history":chat_history})

chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=ai_msg_1["answer"]),
    ]
)

second_ques="What was my previous question about?"
ai_msg_2=rag1_chain.invoke({"input":second_ques,"chat_history":chat_history})
print(ai_msg_2["answer"])


Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=bc83e0d0-ea10-43da-a946-6f7c292874d2,id=bc83e0d0-ea10-43da-a946-6f7c292874d2; trace=bc83e0d0-ea10-43da-a946-6f7c292874d2,id=3bb48c51-6fcb-445d-87dd-ea366a9a7807; trace=bc83e0d0-ea10-43da-a946-6f7c292874d2,id=494841f6-9b2d-48fe-b87e-8ee02af1bc9a; trace=bc83e0d0-ea10-43da-a946-6f7c292874d2,id=52482950-bf20-4937-b1e5-c85fa05031e9; trace=bc83e0d0-ea10-43da-a946-6f7c292874d2,id=6bcff6db-a267-4a84-9c8f-7b44231dcc93; trace=bc83e0d0-ea10-43da-a946-6f7c292874d2,id=032acdf0-f585-465d-aaab-9a567cec5b99; trace=bc83e0d0-ea10-43da-a946-6f7c292874d2,id=313e69ec-cf27-4fa2-af6a-2ec21df4d5ae; trace=bc83e0d0-ea10-43da-a946-6f7c292874d2,id=ff47813f-5336-4323-9af0-5d4f1f871b25; trace=bc83e0d0-ea10-43da-a946-6f7c292874d2,id=afcc925c-4

Your previous question asked for a summary—specifically, “What is this article about?”


Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=9cbd9ca7-9cc0-47a5-9a3d-bac2535dcef0,id=02cb2cf5-6bbd-4c54-a95e-a6e7f350a54b; trace=9cbd9ca7-9cc0-47a5-9a3d-bac2535dcef0,id=fd4147f7-8f07-4455-aab2-e1a14fd3e56f; trace=9cbd9ca7-9cc0-47a5-9a3d-bac2535dcef0,id=c749ee4e-731c-42a5-a744-ac83c071499c; trace=9cbd9ca7-9cc0-47a5-9a3d-bac2535dcef0,id=abff7e50-39de-4c9b-92b0-ac26bf40991f; trace=9cbd9ca7-9cc0-47a5-9a3d-bac2535dcef0,id=228c7cad-409e-4b05-ac26-68c2052d2e46; trace=9cbd9ca7-9cc0-47a5-9a3d-bac2535dcef0,id=fe8a2730-90cc-49d6-b155-01a027ddf7fb; trace=9cbd9ca7-9cc0-47a5-9a3d-bac2535dcef0,id=5901950d-8fa2-41c5-8ea6-de9666d3526a; trace=9cbd9ca7-9cc0-47a5-9a3d-bac2535dcef0,id=3e4d584b-7c4a-43c9-bec0-8d108482b4ed; trace=9cbd9ca7-9cc0-47a5-9a3d-bac2535dcef0,id=72644f16-8

The way of storing history we will use the most

In [26]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

conversational_rag_chain=RunnableWithMessageHistory(
    rag1_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [27]:
conversational_rag_chain.invoke(
    {"input": "What is this article about?"},
    config={
        "configurable": {"session_id": "001"}
    },  # constructs a key "001" in `store`.
)["answer"]

Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=747c5a60-d9ac-4dc9-8043-37c61e5df648; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=30516c1a-0768-4016-b8fe-270da04293df; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=8a7573b7-e9d3-422a-9493-909d8080b750; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=b20eabc5-eb56-4bb2-b683-a8314099c338; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=9fbf8463-4bf7-4242-9449-a365fade0c36; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=56a5c69b-f082-4dbd-bb8f-d9a69f8b975d; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=25febef2-4660-41ed-9090-2c07a82e9b9d; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=581b6276-c4cc-4235-8d16-41904332abb9; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=7e6a83f4-3

'The article is a short essay from Paul Graham’s 2008 Startup School talk that reflects on Y\u202fCombinator’s core advice—“make something people want” and don’t obsess over the business model at first. By juxtaposing those two ideas, Graham humorously notes they describe a charity, then explores whether successful startups can operate like charities, using Craigslist as a prime example. Ultimately, it’s a meditation on startup strategy and the balance between building great products and monetization.'

Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=744ea000-26e1-44f3-8ea7-69c625572276; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=30615f1d-b67e-4625-89ee-b33c092b2cbe; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=d446f13f-b9c2-4a89-890e-598f20d3e1b9; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=56a5c69b-f082-4dbd-bb8f-d9a69f8b975d; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=5eef2203-5dee-42b8-bbec-af19e90c4d8a; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=6402f7f3-e75f-4c45-993d-17f9b50c36e8; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=747c5a60-d9ac-4dc9-8043-37c61e5df648; trace=747c5a60-d9ac-4dc9-8043-37c61e5df648,id=9fbf8463-4bf7-4242-9449-a365fade0c36


In [ ]:
conversational_rag_chain.invoke(
    {"input": "What was my previous question about?"},
    config={"configurable": {"session_id": "001"}},
)["answer"]

Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=0aec63cc-db01-43d9-bde0-2e70280c4812; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=6988cd31-c716-43f4-8ea0-107841ee94e0; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=95fb29e2-8bd2-4a4d-a260-c67ae3294c85; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=c6bac34d-5bab-471f-9a99-d7b2d5a1f408; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=8a4b50bb-83a4-4307-8d9a-bbb24ae1bb42; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=a51a18f8-fd64-4032-8599-c1fa332ac1ba; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=f7166782-8e69-4b15-b334-4f5a499f0f70; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=6ee58f33-d4c4-45fc-83fe-3164d6d24001; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=8c503e5e-d

'Your previous question asked for a summary of what the article is about.'

Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=282b78b8-312b-4309-8849-c0b28b3d8768; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=cdebb514-8257-4a05-977f-d191b3e2b91c; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=407463b6-3d26-45fd-bdc0-7494e109f220; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=a8a7329c-84aa-4856-b33e-fdd8896ffdef; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=fe663d81-65df-4eb5-a706-c52470d39fd0; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=ca04510f-cae8-4726-bc9b-eac2e7aeb8b5; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=844032ae-8cc7-453b-9254-250e3eb5b5f1; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=06f5cedd-6702-4bfa-a778-2e84b8aad2cc; trace=0aec63cc-db01-43d9-bde0-2e70280c4812,id=85f43f0c-c